In [ ]:
import numpy as np
import time

In [ ]:
A = np.array([
        [ 7, 15,  1, 21,  9,  4],
        [18,  2, 12, 10,  6, 16],
        [13, 22,  5,  8, 19,  0],
        [ 3, 11, 14,  7, 20, 15]], dtype=int)

In [ ]:
def rref_mod(A, q):
  R = np.asarray(A, dtype=int, copy=True) % q
  n, m = R.shape

  row = 0
  pivots = []

  for col in range(m):
    pivot = next(
        (i for i in range(row, n) if R[i, col] != 0), None
        )

    if pivot is None:
      continue
    R[[row, pivot]] = R[[pivot, row]]
    inv = pow(int(R[row, col]), -1, q)
    R[row] = (R[row] * inv) % q

    for i in range(n):
      if i != row:
        factor = R[i, col]
        R[i] = (R[i] - factor * R[row]) % q

    pivots.append(col)
    row += 1
    if row == n:
      break

  return R, pivots

In [ ]:
R, pivots = rref_mod(A, 23)
print(A)
print()
print(R)

[[ 7 15  1 21  9  4]
 [18  2 12 10  6 16]
 [13 22  5  8 19  0]
 [ 3 11 14  7 20 15]]

[[ 1  0  0  0  9 20]
 [ 0  1  0  0 18  3]
 [ 0  0  1  0  3  6]
 [ 0  0  0  1 14 13]]


In [ ]:
def kernel_basis_mod(R, pivots, q):
    n, m = R.shape

    free_cols = [j for j in range(m) if j not in pivots]

    K = np.zeros((m, len(free_cols)), dtype=int)

    for k, free_col in enumerate(free_cols):
        K[free_col, k] = 1

        for i, pivot_col in enumerate(pivots):
            K[pivot_col, k] = (-R[i, free_col]) % q

    return K


In [ ]:
K = kernel_basis_mod(R, pivots, 23)

print(K)
print((A @ K) % 23)

[[14  3]
 [ 5 20]
 [20 17]
 [ 9 10]
 [ 1  0]
 [ 0  1]]
[[0 0]
 [0 0]
 [0 0]
 [0 0]]


In [ ]:
def centered_mod(x, q):
    return (x + q // 2) % q - q // 2

In [ ]:

from itertools import product

def find_short_solutions(A, K, q, beta):
    dimension = K.shape[1]
    solutions = []

    for params in product(range(q), repeat=dimension):
        params = np.array(params, dtype=int)

        x = (K @ params) % q
        x = centered_mod(x, q)

        if np.any(x != 0) and np.max(np.abs(x)) <= beta:
            assert np.all((A @ x) % q == 0)
            solutions.append(x)

    return solutions

In [ ]:
def solve_sis(A, q, beta):
    R, pivots = rref_mod(A, q)
    K = kernel_basis_mod(R, pivots, q)
    return find_short_solutions(A, K, q, beta)

In [ ]:
q = 23
beta = 5

start = time.perf_counter()
solutions = solve_sis(A, q, beta)
elapsed = time.perf_counter() - start

print(f"SIS({A.shape[0]}, {A.shape[1]}, {q}, {beta})")
print(f"Solution amount: {len(solutions)}")

if not solutions:
    print("No solutions")

for i, x in enumerate(solutions, start=1):
    print(f"\nSolution: {i}: {tuple(x.tolist())}")
    print(f"Ax mod {q}: {tuple(((A @ x) % q).tolist())}")
    print(f"L_inf:: {np.max(np.abs(x))}")

print(f"\nElapsed time: {elapsed:.6f} с")

SIS(4, 6, 23, 5)
Solution amount: 2

Solution: 1: (-1, -3, 4, 5, 1, -5)
Ax mod 23: (0, 0, 0, 0)
L_inf:: 5

Solution: 2: (1, 3, -4, -5, -1, 5)
Ax mod 23: (0, 0, 0, 0)
L_inf:: 5

Elapsed time: 0.030909 с


In [ ]:
def brute_force_sis(A, q, beta):
  m = A.shape[1]
  solutions = []

  for coords in product(range(-beta, beta + 1), repeat=m):
    if not any(coords):
      continue

    x = np.array(coords, dtype=int)

    if np.all((A @ x) % q == 0):
      solutions.append(x)

  return solutions

In [ ]:
start = time.time()
solutions = brute_force_sis(A, 23, 5)
end = time.time()
print(solutions)
print(end - start)

[array([-1, -3,  4,  5,  1, -5]), array([ 1,  3, -4, -5, -1,  5])]
20.76667857170105
